# ForecastLLM - Week 8 Day 5: Reviewing Forecast Alert Outputs


Donner's Day 5 reviews the app/agent outputs.

ForecastLLM adapts this to reviewing gasoline forecast alert outputs generated earlier in Week 8.

The goal here is human-reviewable operational output: summary metrics, alert records, and notification quality checks.


In [1]:
import json
from pathlib import Path

import pandas as pd

ALERT_THRESHOLD = 0.05
SCHEMA_VERSION = "forecastllm.week8.notification.v1"


In [2]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()), cwd)
RUN_DIR = PROJECT_ROOT / "week8" / "run"

DAY4_SUMMARY_PATH = RUN_DIR / "week8_day4_summary.json"
DAY4_NOTIFICATIONS_PATH = RUN_DIR / "week8_day4_notifications.jsonl"
DAY5_REPORT_PATH = RUN_DIR / "week8_day5_report.md"

missing = [str(p) for p in [DAY4_SUMMARY_PATH, DAY4_NOTIFICATIONS_PATH] if not p.exists()]
if missing:
    missing_text = "\n".join(missing)
    raise FileNotFoundError(
        "Missing required Day 4 artifacts. Run week8/day4.ipynb first.\n"
        f"Missing paths:\n{missing_text}"
    )

print(f"Using summary artifact: {DAY4_SUMMARY_PATH}")
print(f"Using notifications artifact: {DAY4_NOTIFICATIONS_PATH}")


Using summary artifact: /home/geo/Projects/Python/forecastllm/week8/run/week8_day4_summary.json
Using notifications artifact: /home/geo/Projects/Python/forecastllm/week8/run/week8_day4_notifications.jsonl


In [3]:
with DAY4_SUMMARY_PATH.open("r", encoding="utf-8") as f:
    day4_summary = json.load(f)

display_summary = {
    "total_rows": int(day4_summary.get("total_rows", 0)),
    "number_of_alerts": int(day4_summary.get("num_alerts", 0)),
    "increase_alerts": int(day4_summary.get("num_increase", 0)),
    "decrease_alerts": int(day4_summary.get("num_decrease", 0)),
    "threshold": ALERT_THRESHOLD,
}

print("Day 4 summary")
print(json.dumps(display_summary, indent=2))


Day 4 summary
{
  "total_rows": 271,
  "number_of_alerts": 238,
  "increase_alerts": 128,
  "decrease_alerts": 110,
  "threshold": 0.05
}


In [4]:
notifications_df = pd.read_json(DAY4_NOTIFICATIONS_PATH, lines=True)

if notifications_df.empty:
    raise RuntimeError("Day 4 notifications artifact exists but is empty.")

LEGACY_COMPAT_BACKFILL_SCHEMA_VERSION = False

if "schema_version" not in notifications_df.columns:
    if LEGACY_COMPAT_BACKFILL_SCHEMA_VERSION:
        notifications_df["schema_version"] = SCHEMA_VERSION
    else:
        raise ValueError(
            "Missing 'schema_version' in Day 4 notifications. "
            "Re-run week8/day4.ipynb to regenerate artifacts with source schema_version."
        )

if notifications_df["schema_version"].isna().any():
    if LEGACY_COMPAT_BACKFILL_SCHEMA_VERSION:
        notifications_df["schema_version"] = notifications_df["schema_version"].fillna(SCHEMA_VERSION)
    else:
        raise ValueError(
            "Null schema_version values found in Day 4 notifications. "
            "Re-run week8/day4.ipynb to regenerate artifacts with schema_version."
        )

notifications_df["metadata"] = notifications_df["metadata"].apply(lambda x: x if isinstance(x, dict) else {})
notifications_df["timestamp"] = notifications_df["metadata"].apply(lambda m: m.get("cutoff_timestamp"))
notifications_df["alert_type"] = notifications_df["metadata"].apply(lambda m: m.get("alert_type"))
notifications_df["forecast_change"] = notifications_df["metadata"].apply(lambda m: m.get("change"))
notifications_df["baseline_used"] = notifications_df["metadata"].apply(lambda m: m.get("baseline_used"))
notifications_df["recipients_text"] = notifications_df["recipients"].apply(
    lambda r: ", ".join(r) if isinstance(r, list) else str(r)
)

review_df = notifications_df[
    ["timestamp", "alert_type", "subject", "recipients_text", "forecast_change", "baseline_used", "schema_version"]
].copy()

review_df.head(12)


,timestamp,alert_type,subject,recipients_text,forecast_change,baseline_used,schema_version
0,2021-03-01,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,forecast-review@example.com,-0.210,lag_52,forecastllm.week8.notification.v1
1,2021-03-08,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,forecast-review@example.com,-0.336,lag_52,forecastllm.week8.notification.v1
2,2021-03-15,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,forecast-review@example.com,-0.523,lag_52,forecastllm.week8.notification.v1
3,2021-03-22,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,forecast-review@example.com,-0.733,lag_52,forecastllm.week8.notification.v1
4,2021-03-29,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,forecast-review@example.com,-0.860,lag_52,forecastllm.week8.notification.v1
5,2021-04-05,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,forecast-review@example.com,-0.928,lag_52,forecastllm.week8.notification.v1
6,2021-04-12,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,forecast-review@example.com,-1.004,lag_52,forecastllm.week8.notification.v1
7,2021-04-19,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,forecast-review@example.com,-1.037,lag_52,forecastllm.week8.notification.v1
8,2021-04-26,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,forecast-review@example.com,-1.082,lag_52,forecastllm.week8.notification.v1
9,2021-05-03,decrease_alert,[GASREGW] decrease_alert: forecast decrease of...,forecast-review@example.com,-1.083,lag_52,forecastllm.week8.notification.v1


In [5]:
top3_df = review_df.copy()
top3_df["abs_change"] = pd.to_numeric(top3_df["forecast_change"], errors="coerce").abs()
top3_df = top3_df.sort_values("abs_change", ascending=False).head(3)

sample_body = str(notifications_df.iloc[0]["body"])

report_lines = [
    "# Week 8 Day 5 Forecast Alert Review",
    "",
    "## Headline Summary",
    f"- Total rows evaluated: {display_summary['total_rows']}",
    f"- Alerts generated: {display_summary['number_of_alerts']}",
    f"- Increase alerts: {display_summary['increase_alerts']}",
    f"- Decrease alerts: {display_summary['decrease_alerts']}",
    f"- Fixed alert threshold: {display_summary['threshold']}",
    f"- Notification schema version: {SCHEMA_VERSION}",
    "",
    "## Top 3 Alerts (by absolute forecast change)",
]

for i, row in enumerate(top3_df.itertuples(index=False), start=1):
    report_lines.append(
        f"{i}. [{row.alert_type}] {row.subject} | change={row.forecast_change} | baseline={row.baseline_used}"
    )

report_lines.extend(
    [
        "",
        "## Sample Notification Body",
        "```text",
        sample_body,
        "```",
        "",
        "## Next-Step TODOs",
        "- Confirm recipients and routing policy by alert type.",
        "- Add delivery status tracking once sending is introduced.",
        "- Define how this report rolls into week8/results.ipynb.",
    ]
)

report_md = "\n".join(report_lines)
print(report_md[:2000])


# Week 8 Day 5 Forecast Alert Review

## Headline Summary
- Total rows evaluated: 271
- Alerts generated: 238
- Increase alerts: 128
- Decrease alerts: 110
- Fixed alert threshold: 0.05
- Notification schema version: forecastllm.week8.notification.v1

## Top 3 Alerts (by absolute forecast change)
1. [decrease_alert] [GASREGW] decrease_alert: forecast decrease of $1.946 on 2022-06-20 | change=-1.9460000000000002 | baseline=lag_52
2. [decrease_alert] [GASREGW] decrease_alert: forecast decrease of $1.871 on 2022-06-27 | change=-1.870999999999999 | baseline=lag_52
3. [decrease_alert] [GASREGW] decrease_alert: forecast decrease of $1.807 on 2022-06-13 | change=-1.807 | baseline=lag_52

## Sample Notification Body
```text
ForecastLLM Gasoline Alert

Series/commodity: GASREGW (U.S. Regular All Formulations Gasoline Price (Weekly))
Timestamp: 2021-03-01
Current/last observed price: $2.633 dollars_per_gallon
Forecast price: $2.423 dollars_per_gallon
Forecast change: $-0.210 dollars_per_gallon
A

In [6]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
DAY5_REPORT_PATH.write_text(report_md, encoding="utf-8")
print(f"Wrote markdown report -> {DAY5_REPORT_PATH}")


Wrote markdown report -> /home/geo/Projects/Python/forecastllm/week8/run/week8_day5_report.md


This notebook intentionally does not call models, launch UIs, send email, or rebuild earlier pipeline steps.

It is a local review checkpoint for Week 8 outputs before rolling up into `week8/results.ipynb`.
